## Textual Analysis of Facebook and X data on vaccine-related posts by Arabic users.
### Sentiment analysis. 

### Dependency - The analysis utilizes CAMeL Tools
### The Computational Approaches to Modeling Language (CAMeL) Tools is a suite of Arabic natural language processing tools developed by the CAMeL Lab at New York University Abu Dhabi.
#### pip3 install camel_tools

In [ ]:
# import os
# os.environ["CAMELTOOLS_DATA"] = "/your_path/.camel_tools/data"

from camel_tools.utils.normalize import normalize_alef_maksura_ar
from camel_tools.utils.normalize import normalize_alef_ar
from camel_tools.utils.normalize import normalize_teh_marbuta_ar
from camel_tools.utils.normalize import normalize_unicode
from camel_tools.utils.dediac import dediac_ar
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer
from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.tokenizers.morphological import MorphologicalTokenizer
import arabicstopwords.arabicstopwords as stp


import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ar_wordcloud import ArabicWordCloud
import nltk
import datetime
from collections import Counter

text = 'ﷺ'

sentence = "sentence from tweet"

sent_norm = normalize_unicode(sentence)
    

# Normalize alef variants to 'ا'
sent_norm = normalize_alef_ar(sentence)

# Normalize alef maksura 'ى' to yeh 'ي'
sent_norm = normalize_alef_maksura_ar(sent_norm)

# Normalize teh marbuta 'ة' to heh 'ه'
sent_norm = normalize_teh_marbuta_ar(sent_norm)


In [ ]:
# Load the morphological database.
# The MorphologyDB database is used for analyzing modern Standard Arabic. 
db = MorphologyDB.builtin_db()

analyzer = Analyzer(db)

analyses = analyzer.analyze('موظف')

## Import raw data
#### Tweets using the keywords “لقاحات” or “تطعيم” or “لقاح” or “تطعيمات”  - ”vaccines”, “inoculation”, ”vaccine”, “inoculations”
#### FB data are pages and groups collected using keywords 

In [ ]:
# Create dataframe for each corpus
df_fb  = pd.read_csv('your_path/your_file.csv', index_col=0, dtype='unicode')
df_X   = pd.concat( [ pd.read_csv('your_path/file_1.csv', index_col=0)
                             ,pd.read_csv('your_path/file_2.csv', index_col=0)
                             ,pd.read_csv('your_path/file_3.csv', index_col=0)
                             ,pd.read_csv('your_path/file_4.csv', index_col=0) ] )


In [ ]:
# Strip whitespace from the Arabic words 
df_X['Text'] = df_X.Text.apply( lambda x: ' '.join( re.findall(r'[\u0600-\u06FF]+', str(x).strip() ) ) )
df_fb['Text'] = df_fb.Text.apply( lambda x: ' '.join( re.findall(r'[\u0600-\u06FF]+', str(x).strip() ) ) )


In [ ]:
# Print each corpus dataframe size
print(' Twitter Records: ' + str(df_X.size));
print('Facebook Records: ' + str(df_fb.size));


In [ ]:
# Remove empty Text rows for X corpus
df_X.Text = df_X.Text.replace( '', np.nan )
df_X.dropna( subset=['Text'], inplace = True )

# Format the Date for X corpus
for i in range(0,len(df_X['Date'])):
    df_X['Date'].iloc[i]=datetime.datetime.strptime(str(df_X['Date'].iloc[i][:12]).strip(),"%b %d, %Y")



In [ ]:
# Remove empty Text rows for Facebook corpus
df_fb.Text = df_fb.Text.replace( '', np.nan )
df_fb.dropna( subset=['Text'], inplace = True )

# Format the Date for Facebook corpus
for i in range(0,len(df_fb['Date'])):
    df_fb['Date'].iloc[i]=datetime.datetime.strptime(df_fb['Date'].iloc[i],format("%m/%d/%y"))



In [ ]:
# Convert data frames into CSV files
df_X.to_csv('csv/df_X.csv')
df_fb.to_csv('csv/df_fb.csv')

# Process AR Words

In [ ]:
# Add Arabic stopwords from this file
my_stopwords =[]
with open('/your_path/arabic_stopwords_file.txt', 'r') as file:
    for line in (line.rstrip('\n') for line in file):
        my_stopwords.append(line)
#print( my_stopwords)

In [ ]:
import time
from nltk.stem.isri import ISRIStemmer
import unicodedata as ud
from camel_tools.utils.normalize import normalize_unicode

import arabicstopwords.arabicstopwords as stp
nltk_stopwords = nltk.corpus.stopwords.words('arabic')

# Remove stopwords and all characters that are not arabic letters or # numbers and lemmatize the words
def preprocess_ar( text ):
    processedText = []

    my_stp = [stp.stopwords_list() ]+ [u'Arabic_word' ] + nltk_stopwords +  my_stopwords

    # Use Lemmatizer and Stemmer.
    st = ISRIStemmer()
    
    tokenizer = MorphologicalTokenizer(mle, scheme='d3tok', split=True, diac=True)
    
# Use tokenizer to split sentences into words
    tokens = tokenizer.tokenize( text )
    for t in tokens:
        commentwords = ''
        for word in t.split():
            # Checking if the word is a stopword.
            if word not in my_stp and word != ' ':
                if len(word)>1:
                    # Lemmatizing the word.
                    #commentwords += ( st.suf32( word ) + ' ')
                    commentwords += word  + ' '
        processedText.append( normalize_unicode( commentwords ) )
    
    return processedText


In [ ]:

mle = MLEDisambiguator.pretrained('calima-msa-r13')

t = time.time()
# Process the text of each corpus to obtain a bag of words without stopwords ready for visualization
processed_ar_X = preprocess_ar( df_X[ 'Text' ] )
processed_ar_fb = preprocess_ar( df_fb[ 'Text' ] )


In [ ]:
# Write the processed data into a text file for each corpus
file_path='/your_path'

try:
    with open( file_path + 'processed_ar_X.txt', 'w') as f:
        for line in processed_ar_X:
            f.write(f"{line}\n")

    with open( file_path + 'processed_ar_fb.txt', 'w') as f:
        for line in processed_ar_fb:
            f.write(f"{line}\n")

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
              

In [ ]:

## Save sentiments and location to a file. Location is either X or Facebook

import pandas as pd 
import re
# Add sentiments to the overall data frame
df_S  = pd.read_csv('/your_path/sentiments_file.csv', index_col=0, dtype='unicode')
loc = pd.read_csv('/your_path/locations_file.csv', index_col=0, dtype='unicode')

# Merge location with the data frame
df_S = df_S.merge( loc, on='City' )   

#Unify date format in the data frame
def get_year_month( row ):
    return( row.Date[:7])

def fix_date( row ):
    return( row.Date[:10])
    
df_S['YearMonth']  = df_S.apply( get_year_month, axis=1)
df_S['Date']  = df_S.apply( fix_date, axis=1)


In [ ]:
# Save data frame into a CSV file
df_S.to_csv('/Your_path/ready_for_analysis.csv');